# 🤖 AirSense — Retrieval-Augmented Generation (RAG) Health Advisor
**MSc Business Intelligence & Analytics | Capstone Project (Phase 4: Weeks 7 & 8)**

This notebook demonstrates the **AirSense RAG Knowledge Engine & Local GenAI Health Advisor**.

### Core Academic & Technical Objectives:
1. **Hallucination Prevention**: Chaining a local LLM to official WHO 2021 and CPCB NAAQS 2009 guidelines.
2. **Vector Space Embeddings**: Chunking and indexing regulatory standards into a 384-dimensional dense vector store (`all-MiniLM-L6-v2` + ChromaDB).
3. **Semantic Similarity Retrieval**: Evaluating top-K chunk retrieval across diverse vulnerable health personas (asthma, cardiovascular, pediatric, geriatric).
4. **Dynamic Context Augmentation**: Fusing retrieved static guidelines with real-time urban telemetry and 48-hour forecasts.
5. **Edge LLM Inference**: Executing local quantized inference (`llama3.2:3b` via Ollama) with strict safety and medical disclaimer guardrails.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

# Ensure project root is available
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from rag.knowledge_documents import WHO_GUIDELINES, CPCB_GUIDELINES, AQI_HEALTH_ADVISORY
from rag.build_knowledge_base import chunk_text, CHROMA_DB_PATH, COLLECTION_NAME
from rag.health_advisor import (
    retrieve_relevant_context,
    build_rag_prompt,
    ask_health_advisor,
    is_ollama_running,
    is_knowledge_base_ready,
)

print("AirSense RAG modules successfully imported.")
print(f"ChromaDB Ready: {is_knowledge_base_ready()}")
print(f"Ollama LLM Service Running: {is_ollama_running()}")

## 1. Regulatory Knowledge Base Corpus Overview
The AirSense knowledge base aggregates authoritative, peer-reviewed air quality standards and clinical risk thresholds.

In [ ]:
corpus_summary = pd.DataFrame([
    {"Source": "WHO Air Quality Guidelines (2021)", "Characters": len(WHO_GUIDELINES), "Scope": "Global limits (PM2.5: 5 µg/m³ annual, 15 µg/m³ 24h)"},
    {"Source": "CPCB NAAQS India (2009)", "Characters": len(CPCB_GUIDELINES), "Scope": "Indian 6-tier AQI categories & GRAP emergency actions"},
    {"Source": "AirSense Clinical Health Advisory Q&A", "Characters": len(AQI_HEALTH_ADVISORY), "Scope": "Vulnerable population guidelines (asthma, pregnancy, sports, masks)"},
])
corpus_summary

## 2. Document Chunking & Overlap Analysis
To maintain semantic coherence without cutting sentences in half, we apply a sliding window of 600 characters with a 100-character overlap.

In [ ]:
who_chunks = chunk_text(WHO_GUIDELINES, chunk_size=600, overlap=100)
cpcb_chunks = chunk_text(CPCB_GUIDELINES, chunk_size=600, overlap=100)
advisory_chunks = chunk_text(AQI_HEALTH_ADVISORY, chunk_size=600, overlap=100)

print(f"Total Chunks Generated: {len(who_chunks) + len(cpcb_chunks) + len(advisory_chunks)}")
print(f"- WHO Guidelines Chunks: {len(who_chunks)}")
print(f"- CPCB NAAQS Chunks:     {len(cpcb_chunks)}")
print(f"- Health Advisory Chunks: {len(advisory_chunks)}")

# Sample chunk inspection
print("\n--- Sample Chunk (WHO 2021) ---")
print(who_chunks[0][:300] + "...")

## 3. Semantic Similarity Search in Vector Space (ChromaDB)
We query the indexed collection with real-world patient and citizen questions to examine cosine similarity distances and retrieval precision.

In [ ]:
test_queries = [
    "I have asthma and Delhi AQI is 180. Is it safe to jog outdoors?",
    "Should schools suspend outdoor sports when AQI exceeds 250?",
    "Which mask is clinically effective against PM2.5 particles?",
    "What are the WHO safe limits for PM2.5 and PM10 annual exposure?",
]

for q in test_queries:
    results = retrieve_relevant_context(q, top_k=2)
    print(f"\n🔍 Query: '{q}'")
    for idx, r in enumerate(results, 1):
        print(f"   Rank {idx} | Source: {r['source']} | Distance: {r['distance']}")
        snippet = r['text'].replace('\n', ' ')[:120]
        print(f"   Excerpt: \"{snippet}...\"")

## 4. Grounded Prompt Construction (RAG Fusion)
Demonstrating how the prompt is assembled by interleaving official retrieved chunks with live sensor readings.

In [ ]:
sample_q = "Is it safe for an elderly heart patient to take an evening walk in Ahmedabad today?"
retrieved = retrieve_relevant_context(sample_q, top_k=3)
mock_live_aqi = "- Ahmedabad: AQI = 165, PM2.5 = 82.4 µg/m³, Category = Moderate/Poor"

prompt = build_rag_prompt(sample_q, retrieved, live_context=mock_live_aqi)
print("Constructed RAG Prompt (first 500 chars):")
print(prompt[:500] + "\n[... truncated for display ...]")

## 5. End-to-End Grounded Generation via Ollama (llama3.2:3b)
Executing the full RAG pipeline locally with medical disclaimers and authoritative source attribution.

In [ ]:
if is_ollama_running():
    response = ask_health_advisor(
        question="I have asthma, Delhi AQI is 180, can I jog tomorrow?",
        live_aqi_context="- Delhi: AQI = 180 (Moderate/Poor), PM2.5 = 92.0 µg/m³",
    )
    print(f"Model Used: {response['model_used']}")
    print(f"Sources Cited: {response['sources']}")
    print(f"Chunks Retrieved: {response['chunks_retrieved']}")
    print("\n--- AI Health Advisor Response ---")
    print(response["answer"])
else:
    print("Ollama is not running. Start with 'ollama serve' to execute live neural inference.")